In [ ]:
import numpy as np
import h5py
import random
from tqdm.auto import tqdm

from learner_models import OrigUNet

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

import os
from datetime import datetime

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
h5_train_path = "../../../data/real_3.h5"
h5_val_path = "../../../data/real_1.h5"

In [ ]:
checkpoint_dir = "../../../models/finetunes"
!mkdir -p $checkpoint_dir

In [ ]:
#weights_path = "../../../models/pretrained_evfly/sim_forest_Dtheta.pth"
weights_path = "../../../models/finetunes/2026-08-20_19-59-42/checkpoint_epoch_10.pth"

In [ ]:
model = OrigUNet(num_in_channels=1, num_out_channels=1, num_recurrent=[1, 0], enc_params=None, dec_params=None, input_shape=[1, 1, 260, 346], device=device, logger=None, velpred=0, fc_params=None, form_BEV=2, is_deployment=False, is_large=False, evs_min_cutoff=0, skip_type="interp")
model = model.to(device)

In [ ]:
state_dict = torch.load(weights_path, map_location=device, weights_only=True)
#model.load_state_dict(state_dict)
model.load_state_dict(state_dict["model_state_dict"])
"""
def init_weights(m):
    if isinstance(m, nn.Linear):
        # Use Kaiming Normal because we are using ReLU activation
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)

model.apply(init_weights)
"""
model.eval()

In [ ]:
class H5ReadWrapper:
    def __init__(self, path):
        self.path = path

        with h5py.File(path, "r") as f:
            self.len = len(f.keys())

    def __len__(self):
        return self.len

    def __iter__(self):
        with h5py.File(self.path, "r") as f:
            traj_ids = list(f.keys())
            
            random.shuffle(traj_ids)
    
            for traj_id in traj_ids:
                traj = f[traj_id]
                yield traj_id, (traj["trajlength"][()] - 1, traj["depths"][1:], traj["evs"][:])

In [ ]:
enumerate_frames = lambda traj: ((depth, ev) for depth, ev in zip(*traj[1:]))
prep_image = lambda im: torch.from_numpy(im).to(device).unsqueeze(0).unsqueeze(0)

In [ ]:
def calculate_loss(gt, pred, eps=1e-8):
    mask = gt > 0

    valid_gt = gt[mask]
    valid_pred = pred[mask]

    pixel_loss = ((valid_gt - valid_pred) ** 2) / (valid_gt + eps)
    return pixel_loss.mean().cpu(), pixel_loss.median().cpu()

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.0001)

start_epoch = state_dict["epoch"] + 1
num_epochs = 50

In [ ]:
global_step = 0

writer = SummaryWriter(log_dir="runs/")

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

cur_checkpoints = f"{checkpoint_dir}/{timestamp}"
print(f"Saving checkpoints to {cur_checkpoints}")

os.makedirs(cur_checkpoints, exist_ok=False)

for epoch in range(start_epoch, num_epochs):
    model.train()

    train_loss = 0.0
    train_frames = 0

    for traj_name, traj in tqdm(H5ReadWrapper(h5_train_path)):
        h = None

        for i, (depth, ev) in enumerate(enumerate_frames(traj)):
            depth_prep = prep_image(depth)
            ev_prep = prep_image(ev)

            depth_prep = F.interpolate(depth_prep, size=(260, 346), mode='bilinear', align_corners=False)
            ev_prep = F.interpolate(ev_prep, size=(260, 346), mode='bilinear', align_corners=False)

            optimizer.zero_grad()

            _, (depth_pred, _, h) = model([ev_prep, None, h])

            loss, _ = calculate_loss(depth_prep, depth_pred)

            loss.backward()
            optimizer.step()

            h = ([[h[0][0][0].detach(), h[0][0][1].detach()]], None)

            writer.add_scalar("Train/Step_Loss", loss, global_step)

            train_loss += loss

            global_step += 1
            train_frames += 1
    
    avg_train_loss = train_loss / train_frames

    writer.add_scalar("Train/Epoch_Loss", avg_train_loss, epoch)

    model.eval()

    val_loss = 0.0
    val_frames = 0

    with torch.no_grad():
        for traj_name, traj in tqdm(H5ReadWrapper(h5_val_path)):
            h = None

            for i, (depth, ev) in enumerate(enumerate_frames(traj)):
                depth_prep = prep_image(depth)
                ev_prep = prep_image(ev)
    
                depth_prep = F.interpolate(depth_prep, size=(260, 346), mode='bilinear', align_corners=False)
                ev_prep = F.interpolate(ev_prep, size=(260, 346), mode='bilinear', align_corners=False)
    
                _, (depth_pred, _, h) = model([ev_prep, None, h])
    
                loss, _ = calculate_loss(depth_prep, depth_pred)
    
                val_loss += loss
    
                val_frames += 1

    avg_val_loss = val_loss / train_frames

    writer.add_scalar("Val/Epoch_Loss", avg_val_loss, epoch)

    writer.add_scalars("Loss_Comparison", {"Train": avg_train_loss, "Val": avg_val_loss}, epoch)

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "avg_train_loss": avg_train_loss,
        "avg_val_loss": avg_val_loss,
    }

    torch.save(checkpoint, f"{cur_checkpoints}/checkpoint_epoch_{epoch+1}.pth")

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

writer.close()